# 🧠 Brain Tumor MRI Classifier — Refactored CLI Notebook

This is the **command-line execution** notebook for the PyTorch pipeline.
It is designed to be executed non-interactively with Jupyter/nbconvert and to remain an **orchestration-only layer** that matches `ARCHITECTURE.md`.
All training, evaluation, TensorBoard, checkpoint, Grad-CAM, and Flask logic lives in `notebooks/brain_tumor/`. 


## 0 · Install / Refresh the Project Environment

This cell installs the package in editable mode from the repository root and pulls notebook extras.
Run it once per new environment, or skip it if the environment is already prepared from the shell.


In [1]:
from pathlib import Path
import subprocess, sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else CWD.parent
REQ_FILE = REPO_ROOT / 'requirements.txt'

if REQ_FILE.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ_FILE)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_ROOT}[dev,notebook]'])
print(f'✅ Environment refreshed from: {REPO_ROOT}')


✅ Environment refreshed from: /Users/mondo/work/brain-tumor-classifier


## 1 · Configuration

Imports every shared constant from `brain_tumor.config`, then creates the project directories and seeds the run.


In [2]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else CWD.parent
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
for p in [REPO_ROOT, NOTEBOOKS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from brain_tumor import config as cfg

DEVICE = cfg.DEVICE
HAS_CUDA = cfg.HAS_CUDA
HAS_MPS = cfg.HAS_MPS
USE_AMP = cfg.USE_AMP
BASE_DIR = cfg.BASE_DIR
DATA_DIR = cfg.DATA_DIR
MODEL_DIR = cfg.MODEL_DIR
REPORT_DIR = cfg.REPORT_DIR
LOG_DIR = cfg.LOG_DIR
TB_LOG_DIR = cfg.TB_LOG_DIR
CKPT_S1 = cfg.CKPT_S1
CKPT_S2 = cfg.CKPT_S2
FINAL_PATH = cfg.FINAL_PATH
TRAIN_ROOT = getattr(cfg, 'TRAIN_ROOT', DATA_DIR / 'Training')
TEST_ROOT = getattr(cfg, 'TEST_ROOT', DATA_DIR / 'Testing')
CLASS_NAMES = cfg.CLASS_NAMES
NUM_CLASSES = cfg.NUM_CLASSES
DATASET_SLUG = cfg.DATASET_SLUG
IMG_SIZE = cfg.IMG_SIZE
IMG_MEAN = cfg.IMG_MEAN
IMG_STD = cfg.IMG_STD
LABEL_MAP = cfg.LABEL_MAP
BATCH_SIZE = cfg.BATCH_SIZE
DROPOUT = cfg.DROPOUT
SEED = cfg.SEED
EPOCHS_S1 = cfg.EPOCHS_S1
LR_S1 = cfg.LR_S1
EPOCHS_S2 = cfg.EPOCHS_S2
LR_S2 = cfg.LR_S2
UNFREEZE_N = cfg.UNFREEZE_N
NUM_WORKERS = cfg.NUM_WORKERS
PIN_MEMORY = cfg.PIN_MEMORY
PERSISTENT_WORKERS = cfg.PERSISTENT_WORKERS
FLASK_HOST = cfg.FLASK_HOST
FLASK_PORT = cfg.FLASK_PORT
seed_everything = cfg.seed_everything
make_dirs = cfg.make_dirs

make_dirs()
seed_everything()

import torch
print(f'Device   : {DEVICE}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {HAS_CUDA}' + (f'  ({torch.cuda.get_device_name(0)})' if HAS_CUDA else ''))
print(f'MPS      : {HAS_MPS}')
print(f'AMP      : {USE_AMP}')
print(f'BASE_DIR : {BASE_DIR}')
print(f'TRAIN_ROOT: {TRAIN_ROOT}')
print(f'TEST_ROOT : {TEST_ROOT}')


Device   : mps
PyTorch  : 2.11.0
CUDA     : False
MPS      : True
AMP      : False
BASE_DIR : /Users/mondo/work
TRAIN_ROOT: /Users/mondo/work/data/brain_tumor_mri/Training
TEST_ROOT : /Users/mondo/work/data/brain_tumor_mri/Testing


## 2 · Kaggle Authentication

This cell follows the architecture notes: use an existing `~/.kaggle/kaggle.json`, Colab upload, or a local widget upload.


In [3]:
from pathlib import Path
import json, os, sys
import ipywidgets as widgets
from IPython.display import display, clear_output

KAGGLE_DIR  = Path.home() / '.kaggle'
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

def _save_creds(raw: bytes) -> str:
    creds = json.loads(raw.decode('utf-8'))
    assert 'username' in creds and 'key' in creds, 'Invalid kaggle.json format'
    KAGGLE_JSON.write_text(json.dumps(creds))
    os.chmod(KAGGLE_JSON, 0o600)
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']
    return creds['username']

if KAGGLE_JSON.exists():
    creds = json.loads(KAGGLE_JSON.read_text())
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']
    print(f'✅ Using existing kaggle.json for: {creds["username"]}')

elif 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        user = _save_creds(uploaded['kaggle.json'])
        print(f'✅ Uploaded kaggle.json for: {user}')
    else:
        print('⚠️ Upload kaggle.json to enable dataset download.')

else:
    uploader = widgets.FileUpload(accept='.json', multiple=False)
    out = widgets.Output()

    def _on_upload(change):
        if not uploader.value:
            return
        file_info = list(uploader.value)[0] if isinstance(uploader.value, dict) else uploader.value[0]
        raw = file_info['content'] if isinstance(file_info, dict) else file_info.content.tobytes()
        with out:
            clear_output(wait=True)
            try:
                user = _save_creds(raw)
                print(f'✅ Saved kaggle.json for: {user}')
            except Exception as exc:
                print(f'❌ Kaggle credential error: {exc}')

    uploader.observe(_on_upload, names='value')
    display(widgets.VBox([widgets.HTML('<b>Upload kaggle.json</b>'), uploader, out]))


✅ Using existing kaggle.json for: mondosoria


## 3 · Download & Inspect Dataset


In [ ]:
from pathlib import Path
import importlib
import brain_tumor.config as cfg_mod
import brain_tumor.data.dataset as ds

# Ensure notebook kernel uses latest local module code (avoid stale cached imports).
importlib.reload(cfg_mod)
importlib.reload(ds)
download_dataset = ds.download_dataset

# Resolve repo root from loaded config module instead of relying on previous cells/cwd.
cfg_path = Path(cfg_mod.__file__).resolve()
module_repo_root = next(
    (p for p in [cfg_path.parent, *cfg_path.parent.parents] if (p / 'pyproject.toml').exists()),
    Path.cwd().resolve(),
)

# Resolve data directory robustly.
candidate_dirs = [
    Path(cfg_mod.DATA_DIR),
    module_repo_root / 'data' / 'brain_tumor_mri',
    module_repo_root / 'data',
    Path.cwd().resolve() / 'data' / 'brain_tumor_mri',
    Path.cwd().resolve() / 'data',
]
resolved_data_dir = next((p for p in candidate_dirs if p.exists()), Path(cfg_mod.DATA_DIR))

print(f'Using config module : {cfg_path}')
print(f'Using dataset module: {Path(ds.__file__).resolve()}')
print(f'Using DATA_DIR      : {resolved_data_dir.resolve()}')

TRAIN_ROOT, TEST_ROOT = download_dataset(resolved_data_dir, cfg_mod.DATASET_SLUG)
print(f'Train root exists: {TRAIN_ROOT.exists()}')
print(f'Test  root exists: {TEST_ROOT.exists()}')
print(f'Classes         : {cfg_mod.CLASS_NAMES}')
print(f'Label map keys  : {sorted(cfg_mod.LABEL_MAP.keys())}')


Using dataset module: /Users/mondo/work/brain-tumor-classifier/brain_tumor/data/dataset.py
Using DATA_DIR      : /Users/mondo/work/data/brain_tumor_mri
Dataset already present ✔


FileNotFoundError: Could not locate dataset split folders under /Users/mondo/work/data/brain_tumor_mri. Expected Training/Testing (any case). Top-level entries: []

## 4 · Dataset, Augmentation & DataLoaders


In [ ]:
from brain_tumor.data.dataset import build_dataloaders, get_val_transform
from brain_tumor.evaluation.plots import plot_augmented_samples

train_loader, val_loader, test_loader, full_train_aug = build_dataloaders(
    TRAIN_ROOT, TEST_ROOT
)
val_transform = get_val_transform()

print(f'Train : {len(train_loader.dataset):>5} images  ({len(train_loader)} batches)')
print(f'Val   : {len(val_loader.dataset):>5} images  ({len(val_loader)} batches)')
print(f'Test  : {len(test_loader.dataset):>5} images  ({len(test_loader)} batches)')
print(f'pin_memory={PIN_MEMORY}  num_workers={NUM_WORKERS}  persistent_workers={PERSISTENT_WORKERS}')
plot_augmented_samples(train_loader)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/mondo/work/data/brain_tumor_mri/Training'

## 5 · Class-Weight Computation


In [ ]:
from brain_tumor.evaluation.metrics import compute_class_weights

class_weights = compute_class_weights(
    full_train_aug,
    train_loader.dataset.indices,
)
print('Class weights:', class_weights)


## 6 · Model — EfficientNetB0


In [ ]:
from brain_tumor.models.classifier import BrainTumorClassifier

model = BrainTumorClassifier(NUM_CLASSES, DROPOUT).to(DEVICE)
print(f'Total params    : {model.n_total:,}')
print(f'Trainable params: {model.n_trainable:,}')


## 7 · TensorBoard Setup


In [ ]:
from brain_tumor.training.tensorboard import setup_writer, launch_tensorboard

writer = setup_writer(model)
tb_proc = launch_tensorboard()
print(f'📊 TensorBoard logdir: {TB_LOG_DIR}')
print('🛑 Stop later with: tb_proc.terminate()')


## 8 · Stage 1 — Frozen Backbone


In [ ]:
import torch.nn as nn
import torch.optim as optim
from brain_tumor.training.engine import run_stage

model.freeze_backbone()
print(f'Trainable params (Stage 1): {model.n_trainable:,}')

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S1)
scheduler1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer1, mode='min', factor=0.4, patience=4)

history_s1 = run_stage(
    model, train_loader, val_loader,
    criterion, optimizer1, scheduler1,
    EPOCHS_S1, 'S1-frozen', CKPT_S1,
    tb_writer=writer, global_step_offset=0,
)
print(f'✅ Stage 1 complete → {CKPT_S1}')


## 9 · Stage 2 — Fine-tune Last 30 Layers


In [ ]:
import torch

model.load_state_dict(torch.load(CKPT_S1, map_location=DEVICE))
model.unfreeze_last_n(UNFREEZE_N)
print(f'Trainable params (Stage 2): {model.n_trainable:,}')

optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S2)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.4, patience=4)

history_s2 = run_stage(
    model, train_loader, val_loader,
    criterion, optimizer2, scheduler2,
    EPOCHS_S2, 'S2-finetune', CKPT_S2,
    tb_writer=writer,
    global_step_offset=len(history_s1['train_loss']),
)
writer.flush(); writer.close()
print(f'✅ Stage 2 complete → {CKPT_S2}')


## 10 · Training Curves


In [ ]:
from brain_tumor.evaluation.plots import plot_history

plot_history(history_s1, history_s2)


## 11 · Test Evaluation


In [ ]:
from brain_tumor.models.classifier import BrainTumorClassifier
from brain_tumor.evaluation.metrics import get_predictions, print_classification_report

best_model = BrainTumorClassifier(NUM_CLASSES, DROPOUT).to(DEVICE)
best_model.load_state_dict(torch.load(CKPT_S2, map_location=DEVICE))
best_model.eval()

y_true, y_pred, y_prob, img_paths = get_predictions(best_model, test_loader)
print_classification_report(y_true, y_pred)


## 12 · Confusion Matrix


In [ ]:
from brain_tumor.evaluation.plots import plot_confusion_matrix

plot_confusion_matrix(y_true, y_pred)


## 13 · Per-Class ROC Curves


In [ ]:
from brain_tumor.evaluation.plots import plot_roc_curves

roc_scores = plot_roc_curves(y_true, y_prob)
print('ROC scores:', roc_scores)


## 14 · Misclassification Review


In [ ]:
from brain_tumor.evaluation.plots import plot_misclassified
from brain_tumor.evaluation.metrics import build_error_dataframe

errors_df = build_error_dataframe(y_true, y_pred, y_prob, img_paths)
display(errors_df.head(20))
plot_misclassified(errors_df)


## 15 · Grad-CAM Visualisation


In [ ]:
from brain_tumor.evaluation.gradcam import display_gradcam

correct_idx = next(i for i in range(len(y_true)) if y_true[i] == y_pred[i])
display_gradcam(
    img_paths[correct_idx], best_model, val_transform,
    title_prefix='Grad-CAM — Correct Prediction',
)

if len(errors_df) > 0:
    row = errors_df.iloc[0]
    display_gradcam(
        row['path'], best_model, val_transform,
        title_prefix=f'Grad-CAM — MISCLASSIFIED (true: {row["true_label"]}  pred: {row["pred_label"]})',
    )
else:
    print('No misclassified cases to visualise.')


## 16 · Save Model & Metrics


In [ ]:
import numpy as np
from brain_tumor.models.checkpoint import save_model, save_metrics

save_model(best_model, FINAL_PATH)

macro_auc = float(np.mean(list(roc_scores.values())))
metrics_payload = {
    'macro_auc'    : macro_auc,
    'roc_per_class': {k: float(v) for k, v in roc_scores.items()},
    'total_test'   : int(len(y_true)),
    'misclassified': int(len(errors_df)),
}
save_metrics(metrics_payload)
print(metrics_payload)


## 17 · Flask Inference App


In [ ]:
import os, sys, time, subprocess, urllib.request

FLASK_URL    = f'http://{FLASK_HOST}:{FLASK_PORT}'
FLASK_HEALTH = f'{FLASK_URL}/health'
FLASK_LOG    = LOG_DIR / 'flask_server.log'
flask_script = BASE_DIR / 'src' / 'app' / 'app.py'

try:
    if 'flask_proc' in globals() and flask_proc and flask_proc.poll() is None:
        flask_proc.terminate(); time.sleep(1)
except Exception:
    pass

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
log_fh = open(FLASK_LOG, 'w', buffering=1)
flask_proc = subprocess.Popen(
    [sys.executable, '-u', str(flask_script)],
    stdout=log_fh, stderr=subprocess.STDOUT,
    text=True, cwd=str(flask_script.parent), env=env,
)

healthy = False
for _ in range(30):
    if flask_proc.poll() is not None:
        break
    try:
        urllib.request.urlopen(FLASK_HEALTH, timeout=1.5)
        healthy = True
        break
    except Exception:
        time.sleep(0.5)

if healthy:
    print(f'✅ Flask running → {FLASK_URL}')
    print(f'📝 Logs: {FLASK_LOG}')
    print('🛑 Stop later with: flask_proc.terminate()')
else:
    print('❌ Flask did not start. Check logs:')
    print(FLASK_LOG.read_text()[-3000:])


## 18 · Optional · Shutdown


In [ ]:
ENABLE_FLASK_STOP = False
if ENABLE_FLASK_STOP and 'flask_proc' in globals() and flask_proc and flask_proc.poll() is None:
    import time
    flask_proc.terminate()
    time.sleep(1)
    print('Flask stopped.')


In [ ]:
ENABLE_TB_STOP = False
if ENABLE_TB_STOP and 'tb_proc' in globals() and tb_proc and tb_proc.poll() is None:
    import time
    tb_proc.terminate()
    time.sleep(1)
    print('TensorBoard stopped.')


## 19 · Output Artefact Inventory


In [ ]:
import pandas as pd

inventory = [
    {'path': str(p.relative_to(BASE_DIR)), 'size_kb': round(p.stat().st_size / 1024, 2)}
    for root in [MODEL_DIR, REPORT_DIR, LOG_DIR]
    for p in sorted(root.rglob('*')) if p.is_file()
]
display(pd.DataFrame(inventory))


---

This notebook is intentionally thin. If you need to change training logic, update the package modules under `notebooks/brain_tumor/`, not the notebook cells. See `ARCHITECTURE.md` for the module ownership map.
